In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

In [ ]:
### 1. 불필요한 id, product_type, shot 컬럼 제거
df = df.drop(columns=[('process', 'id'), ('process', 'shot'), ('process', 'product_type')])

In [ ]:
### 2. 값이 전부 0인 defects 컬럼 drop
# Defects 하위 컬럼들
defect_cols = df['defects'].columns

# 값이 전부 0인 Defects 컬럼 찾기
zero_defects = [
    col for col in defect_cols
    if df[('defects', col)].sum() == 0
]

# 결과 출력
print("삭제될 defects 컬럼:")
print(zero_defects)

print("\n삭제될 컬럼 수:", len(zero_defects))
print("삭제 전 defects 컬럼 수:", len(defect_cols))

# 실제 삭제
df = df.drop(columns=[('defects', col) for col in zero_defects])

print("삭제 후 defects 컬럼 수:", len(df['defects'].columns))

In [ ]:
df['defects'].sum().sort_values(ascending=False)

In [ ]:
# 불량 범주 정의 (suffix _1, _2 자동 처리)
defect_groups = {
    "표면": [
        "stain_1",
        "dent_1",
        "dent_2",
        "scratch_1",
        "buring_mark_1"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "blow_hole_1",
        "blow_hole_2",
        "bubble_1",
        "crack_1",
        "crack_2"
    ],

    "이물질": [
        "contamination_1",
        "contamination_2",
        "impurity_1",
        "impurity_2",
        "inclusions_2"
    ]
}

In [ ]:
import numpy as np
import pandas as pd

defects = df['defects'].copy()  # columns: Short_Shot_1, Blow_Hole_2, ...

# 각 범주별로 해당하는 defect 컬럼들을 찾아서 "하나라도 1이면 1"로 만들기
y_group = pd.DataFrame(index=df.index)

for group_name, base_names in defect_groups.items():
    # base_names 중 하나로 시작하는 컬럼들 찾기 (예: "Short_Shot" -> "Short_Shot_1", "Short_Shot_2")
    cols = [c for c in defects.columns if any(str(c).startswith(b) for b in base_names)]
    
    if len(cols) == 0:
        # 해당 범주에 매칭되는 컬럼이 없으면 0으로
        y_group[group_name] = 0
    else:
        y_group[group_name] = (defects[cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)

print(y_group.sum().sort_values(ascending=False))   # 범주별 발생 건수 확인
print(y_group.mean().mul(100))                      # 범주별 발생률(%) 확인

In [ ]:
# 입력 변수
X = df['process'].copy()

# 혹시 결측이 있으면 일단 중앙값으로 채우기
X = X.fillna(X.median())

# Stage 1 타겟: 하나라도 불량이면 1
y_binary = (y_group.sum(axis=1) > 0).astype(int)

print("정상/불량 분포")
print(y_binary.value_counts())
print(y_binary.value_counts(normalize=True))

In [ ]:
X_train, X_test, y_train_bin, y_test_bin, y_train_group, y_test_group = train_test_split(
    X, y_binary, y_group,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(X_train.shape, X_test.shape)

In [ ]:
# SMOTE 적용
smote_stage1 = SMOTE(random_state=42)
X_train_sm1, y_train_bin_sm = smote_stage1.fit_resample(X_train, y_train_bin)

print("SMOTE 후 Stage1 분포")
print(pd.Series(y_train_bin_sm).value_counts())

# Stage 1 모델
stage1_model = XGBClassifier(
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

stage1_model.fit(X_train_sm1, y_train_bin_sm)

In [ ]:
# 불량 확률
stage1_proba = stage1_model.predict_proba(X_test)[:, 1]

# threshold 조정
threshold = 0.25
stage1_pred = (stage1_proba >= threshold).astype(int)

print("=== Stage 1: 불량 여부 분류 ===")
print(classification_report(y_test_bin, stage1_pred, zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test_bin, stage1_pred))

In [ ]:
# 단일 불량만 있는 행만 사용
single_defect_mask = (y_group.sum(axis=1) == 1)

# 대표 불량 유형
y_type = y_group.idxmax(axis=1)

# Stage 2 전체 후보
X_type = X[single_defect_mask]
y_type = y_type[single_defect_mask]

print("Stage 2 학습 후보 분포")
print(y_type.value_counts())
print(y_type.value_counts(normalize=True))

In [ ]:
print("전체 불량 데이터 수:", (y_group.sum(axis=1) > 0).sum())
print("단일 불량 데이터 수:", single_defect_mask.sum())

In [ ]:
# Stage 1 split 기준으로 Stage 2 train/test 구성
train_idx = X_train.index
test_idx = X_test.index

stage2_train_mask = single_defect_mask.loc[train_idx]
stage2_test_mask = single_defect_mask.loc[test_idx]

X_train_stage2 = X_train[single_defect_mask.loc[X_train.index]]
y_train_stage2 = y_type.loc[X_train_stage2.index]

X_test_stage2 = X_test[single_defect_mask.loc[X_test.index]]
y_test_stage2 = y_type.loc[X_test_stage2.index]

print("Stage 2 train shape:", X_train_stage2.shape)
print("Stage 2 test shape:", X_test_stage2.shape)
print("\nStage 2 train class 분포")
print(y_train_stage2.value_counts())

In [ ]:
# 문자열 라벨 → 숫자 라벨
label_map = {'표면': 0, '구조': 1, '이물질': 2}
inv_label_map = {v: k for k, v in label_map.items()}

y_train_stage2_num = y_train_stage2.map(label_map)
y_test_stage2_num = y_test_stage2.map(label_map)

# SMOTE
from imblearn.over_sampling import SMOTE

smote_stage2 = SMOTE(random_state=42, k_neighbors=1)
X_train_sm2, y_train_stage2_sm = smote_stage2.fit_resample(X_train_stage2, y_train_stage2_num)

print("\nSMOTE 후 Stage2 분포")
print(pd.Series(y_train_stage2_sm).value_counts())

# XGBoost
from xgboost import XGBClassifier

stage2_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

stage2_model.fit(X_train_sm2, y_train_stage2_sm)

# 예측
stage2_pred_num = stage2_model.predict(X_test_stage2)
stage2_pred = pd.Series(stage2_pred_num, index=X_test_stage2.index).map(inv_label_map)

In [ ]:

print("=== Stage 2: 불량 종류 분류 ===")
print(classification_report(y_test_stage2, stage2_pred, zero_division=0))

smote, xgboost

In [ ]:
X_train_stage2.groupby(y_train_stage2).mean()

In [ ]:
# Stage 1 확률 / 예측
stage1_proba_test = stage1_model.predict_proba(X_test)[:, 1]
stage1_pred_test = (stage1_proba_test >= threshold).astype(int)

# 기본 결과 테이블
final_pred = pd.DataFrame(index=X_test.index)
final_pred["불량확률"] = stage1_proba_test
final_pred["불량여부_예측"] = stage1_pred_test
final_pred["예측_불량종류"] = "정상"

# Stage 1에서 불량으로 예측된 샘플만 Stage 2 분류
pred_defect_idx = final_pred.index[final_pred["불량여부_예측"] == 1]

if len(pred_defect_idx) > 0:
    stage2_pred_for_defects = stage2_model.predict(X_test.loc[pred_defect_idx])
    final_pred.loc[pred_defect_idx, "예측_불량종류"] = stage2_pred_for_defects

final_pred.head(20)

In [ ]:
actual_result = pd.DataFrame(index=X_test.index)
actual_result["실제_불량여부"] = y_test_bin.values
actual_result["실제_불량종류"] = "정상"

# 실제 단일 불량 범주인 경우만 실제 종류 표시
single_test_idx = y_test_group.index[(y_test_group.sum(axis=1) == 1)]
actual_result.loc[single_test_idx, "실제_불량종류"] = y_test_group.loc[single_test_idx].idxmax(axis=1)

result_compare = final_pred.join(actual_result)
result_compare.head(20)